# collaborative filtering

In [7]:
pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pyspark

In [9]:
pip install findspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Collaborative Filtering Example") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

In [11]:
import time  
import pyspark  
from pyspark.sql import SparkSession  
spark = SparkSession.builder.appName('recommendation').getOrCreate()

In [12]:
movies = spark.read.load("movies.csv", format='csv', header = True)
ratings = spark.read.load('ratings.csv', format='csv', header = True)
links = spark.read.load("links.csv", format='csv', header = True)
tags = spark.read.load("tags.csv", format='csv', header = True)
ratings.show()

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    223|   3.0|964980985|
|     1|    231|   5.0|964981179|
|     1|    235|   4.0|964980908|
|     1|    260|   5.0|964981680|
|     1|    296|   3.0|964982967|
|     1|    316|   3.0|964982310|
|     1|    333|   5.0|964981179|
|     1|    349|   4.0|964982563|
+------+-------+------+---------+
only showing top 20 rows



In [13]:
import time  
import pyspark  
from pyspark.sql import SparkSession  
spark = SparkSession.builder.appName('recommendation').getOrCreate()

movies = spark.read.load("movies.csv", format='csv', header = True)
ratings = spark.read.load('ratings.csv', format='csv', header = True)
links = spark.read.load("links.csv", format='csv', header = True)
tags = spark.read.load("tags.csv", format='csv', header = True)
ratings.show()
ratings=ratings.select("userId","movieId","rating")
ratings.printSchema()
df = ratings.withColumn('userId', ratings['userId'].cast('int')).\
withColumn('movieId', ratings['movieId'].cast('int')).withColumn('rating', ratings['rating'].cast('float'))
df.printSchema()
train, validation, test = df.randomSplit([0.6,0.2,0.2], seed = 0)
print("The number of ratings in each set: {}, {}, {}".format(train.count(), validation.count(), test.count()))

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
|     1|     70|   3.0|964982400|
|     1|    101|   5.0|964980868|
|     1|    110|   4.0|964982176|
|     1|    151|   5.0|964984041|
|     1|    157|   5.0|964984100|
|     1|    163|   5.0|964983650|
|     1|    216|   5.0|964981208|
|     1|    223|   3.0|964980985|
|     1|    231|   5.0|964981179|
|     1|    235|   4.0|964980908|
|     1|    260|   5.0|964981680|
|     1|    296|   3.0|964982967|
|     1|    316|   3.0|964982310|
|     1|    333|   5.0|964981179|
|     1|    349|   4.0|964982563|
+------+-------+------+---------+
only showing top 20 rows

root
 |-- userId: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- rating: string (nullable = true)

root
 |-- userId: integer (nullable =

In [14]:
from pyspark.sql.functions import col, sqrt
def RMSE(predictions):
    squared_diff = predictions.withColumn("squared_diff", pow(col("rating") - col("prediction"), 2))
    mse = squared_diff.selectExpr("mean(squared_diff) as mse").first().mse
    return mse ** 0.5

from pyspark.ml.recommendation import ALS

def GridSearch(train, valid, num_iterations, reg_param, n_factors):
    min_rmse = float('inf')
    best_n = -1
    best_reg = 0
    best_model = None
    for n in n_factors:
        for reg in reg_param:
            als = ALS(rank = n, 
                      maxIter = num_iterations, 
                      seed = 0, 
                      regParam = reg,
                      userCol="userId", 
                      itemCol="movieId", 
                      ratingCol="rating", 
                      coldStartStrategy="drop")            
            model = als.fit(train)
            predictions = model.transform(valid)
            rmse = RMSE(predictions)     
            print('{} latent factors and regularization = {}: validation RMSE is {}'.format(n, reg, rmse))
            if rmse < min_rmse:
                min_rmse = rmse
                best_n = n
                best_reg = reg
                best_model = model
                
    pred = best_model.transform(train)
    train_rmse = RMSE(pred)
    print('\nThe best model has {} latent factors and regularization = {}:'.format(best_n, best_reg))
    print('traning RMSE is {}; validation RMSE is {}'.format(train_rmse, min_rmse))
    return best_model

In [15]:
from pyspark.sql.functions import col, sqrt
num_iterations = 10
ranks = [6, 8, 10, 12]
reg_params = [0.05, 0.1, 0.2, 0.4, 0.8]

start_time = time.time()
final_model = GridSearch(train, validation, num_iterations, reg_params, ranks)
print('Total Runtime: {:.2f} seconds'.format(time.time() - start_time))

6 latent factors and regularization = 0.05: validation RMSE is 0.9774929338436005
6 latent factors and regularization = 0.1: validation RMSE is 0.9129091201278514
6 latent factors and regularization = 0.2: validation RMSE is 0.8951553352727597
6 latent factors and regularization = 0.4: validation RMSE is 0.9694803165579974
6 latent factors and regularization = 0.8: validation RMSE is 1.1934058858997667
8 latent factors and regularization = 0.05: validation RMSE is 0.9911454501618319
8 latent factors and regularization = 0.1: validation RMSE is 0.9168968760403396
8 latent factors and regularization = 0.2: validation RMSE is 0.8984989555521362
8 latent factors and regularization = 0.4: validation RMSE is 0.9702570876695594
8 latent factors and regularization = 0.8: validation RMSE is 1.1934001704790351
10 latent factors and regularization = 0.05: validation RMSE is 0.997857980610933
10 latent factors and regularization = 0.1: validation RMSE is 0.9176672188395238
10 latent factors and re

In [16]:
pred_test = final_model.transform(test)
print('The testing RMSE is ' + str(RMSE(pred_test)))
single_user = test.filter(test['userId']==12).select(['movieId','userId'])
single_user.show()
single_user.join(movies, single_user.movieId == movies.movieId, 'inner').show()
reccomendations = final_model.transform(single_user)
reccomendations.orderBy('prediction',ascending=False).show()
reccomendations.join(movies, reccomendations.movieId == movies.movieId, 'inner').show()

The testing RMSE is 0.8959197518478413
+-------+------+
|movieId|userId|
+-------+------+
|    543|    12|
|   1357|    12|
|   2485|    12|
+-------+------+

+-------+------+-------+--------------------+--------------------+
|movieId|userId|movieId|               title|              genres|
+-------+------+-------+--------------------+--------------------+
|    543|    12|    543|So I Married an A...|Comedy|Romance|Th...|
|   1357|    12|   1357|        Shine (1996)|       Drama|Romance|
|   2485|    12|   2485|She's All That (1...|      Comedy|Romance|
+-------+------+-------+--------------------+--------------------+

+-------+------+----------+
|movieId|userId|prediction|
+-------+------+----------+
|   1357|    12| 5.0159354|
|    543|    12| 3.6550279|
|   2485|    12| 3.4955368|
+-------+------+----------+

+-------+------+----------+-------+--------------------+--------------------+
|movieId|userId|prediction|movieId|               title|              genres|
+-------+------+--

In [17]:
from pyspark.sql.functions import col, lit

user_id = 12
single_user_ratings = test.filter(test['userId'] == user_id).select(['movieId', 'userId', 'rating'])
print("Movies liked by user with ID", user_id)
single_user_ratings.join(movies, 'movieId').select('movieId', 'title', 'rating').show()

Movies liked by user with ID 12
+-------+--------------------+------+
|movieId|               title|rating|
+-------+--------------------+------+
|    543|So I Married an A...|   3.5|
|   1357|        Shine (1996)|   5.0|
|   2485|She's All That (1...|   5.0|
+-------+--------------------+------+



In [18]:

all_movies = df.select('movieId').distinct()
user_movies = single_user_ratings.select('movieId').distinct()
movies_to_recommend = all_movies.subtract(user_movies)

In [19]:
recommendations = final_model.transform(movies_to_recommend.withColumn('userId', lit(user_id)))
recommendations = recommendations.filter(col('prediction') > 0)

In [20]:
print("Recommended movies for user with ID", user_id)
recommended_movies = recommendations.join(movies, 'movieId').select('movieId', 'title', 'prediction')

Recommended movies for user with ID 12


In [21]:
ordered_recommendations = recommended_movies.orderBy(col('prediction').desc())
ordered_recommendations.show()

+-------+--------------------+----------+
|movieId|               title|prediction|
+-------+--------------------+----------+
|  67618|Strictly Sexual (...|  6.166763|
|   3379| On the Beach (1959)|  6.117749|
|   5867|        Thief (1981)| 5.9761686|
|  42730|   Glory Road (2006)| 5.9761686|
|   4535|Man from Snowy Ri...| 5.9761686|
|   7121|   Adam's Rib (1949)| 5.9670253|
|  60943| Frozen River (2008)|  5.941128|
|  33649|  Saving Face (2004)|  5.935265|
|  25906|Mr. Skeffington (...|  5.927385|
|  77846| 12 Angry Men (1997)|  5.927385|
|   3200|Last Detail, The ...|  5.890436|
|   3567|   Bossa Nova (2000)|  5.871085|
|  94070|Best Exotic Marig...|  5.857238|
|   4789|Phantom of the Pa...| 5.8511386|
|   3086|Babes in Toyland ...|  5.848856|
| 138966|Nasu: Summer in A...|  5.843027|
|  26928|Summer's Tale, A ...|  5.843027|
|   3819|      Tampopo (1985)|  5.843027|
|  84273|Zeitgeist: Moving...|  5.843027|
| 184245|De platte jungle ...|  5.843027|
+-------+--------------------+----